# 060 — Round 4: grouped k-fold training (`attention_unet_nll`)

`fixing.md` #4. A genuine held-out **variance** estimate — how much the metrics move
across independent train/val splits — instead of the single fixed split every other
notebook uses.

**Scope** (decided from `C4`/`C5`, `final-comments.md`): **one model**,
`attention_unet_nll` — top-2 on every detection cut and the NLL head covers both
`structural delta` (from its `μ`) and `structural z`. `k = settings.KFOLD_K` (3).
`β = settings.NLL_BETA` (0.5), fixed. A second model is only added if fold variance
turns out alarming.

**Split** (`scripts.kfold`): real artworks partitioned into `k` groups *by artwork
ID* (no section leakage); fold `i` holds out group `i` as validation, the rest **plus
all mockups** are train. No test split — the held-out artworks are the fold's
evaluation set. `data/test/` (GT paintings) is untouched.

**Run it across several short sessions.** Each fold is one `train_single.py`
subprocess that exits when done; the loop below is **skip-if-exists**, so re-running
this notebook trains only the next missing fold. Set `MAX_FOLDS_PER_RUN = 1` to stop
after one fold per session. Realistic per fold on this hardware: ~1–3 h (the NLL runs
in `024` early-stopped around epoch 25; the 100-epoch cap is the upper bound).

Evaluate with `061_kfold_evaluation.ipynb` once ≥ 2 folds exist.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [1]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports.

In [2]:
import json
import subprocess

from scripts.config import settings
from scripts.dataset import load_image_pairs
from scripts.kfold import fold_artwork_groups, grouped_kfold_splits

print(f"KFOLD_K = {settings.KFOLD_K}   KFOLD_SEED = {settings.KFOLD_SEED}   NLL_BETA = {settings.NLL_BETA}")

KFOLD_K = 3   KFOLD_SEED = 42   NLL_BETA = 0.5


## 1. The fold plan

Deterministic in `(KFOLD_K, KFOLD_SEED)` and the set of real-artwork IDs. Mockups are
in every fold's train set.

In [3]:
ARCH = "attention_unet_nll"
K = settings.KFOLD_K
KFOLD_DIR = settings.MODELS_DIR / "kfold"
KFOLD_LOG_DIR = settings.LOGS_DIR / "kfold"

pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
held_out = fold_artwork_groups(pairs, k=K, seed=settings.KFOLD_SEED)
splits = grouped_kfold_splits(pairs, k=K, seed=settings.KFOLD_SEED)

print(f"{len(pairs)} pairs total\n")
for i, (groups, (tr, va)) in enumerate(zip(held_out, splits)):
    print(f"fold {i}: held out {len(groups)} artworks -> val {len(va)} pairs / train {len(tr)} pairs")
    print(f"         {groups}")

1164 pairs total

fold 0: held out 9 artworks -> val 333 pairs / train 831 pairs
         ['mano', 'mod', 'natmorta2', 'q2', 'q3', 'santo', 'sch02', 'sch03', 'volto']
fold 1: held out 8 artworks -> val 221 pairs / train 943 pairs
         ['a1', 'c1', 'cristo', 'natmorta3', 'oblato_tot', 'oblato_volto', 'sch01', 'testa']
fold 2: held out 8 artworks -> val 214 pairs / train 950 pairs
         ['a2', 'b1', 'corpo', 'natmorta1', 'orecchio', 'q1', 'torso', 'veste']


## 2. Train — one subprocess per fold, resumable

`scripts.train_single --fold i --kfold-k K` builds fold `i`'s split itself
(deterministic from `settings`), so nothing crosses the process boundary. Checkpoints
land in `models/kfold/fold_<i>/attention_unet_nll/best_model.keras` (+ `history.json`).

A fold whose checkpoint already exists is skipped. `MAX_FOLDS_PER_RUN` caps how many
folds this run will train — set it to `1` to do exactly one fold per session.

In [4]:
MAX_FOLDS_PER_RUN = 1  # e.g. 1 to train a single fold per session
EPOCHS = settings.EPOCHS

trained_this_run = 0
for fold in range(K):
    model_dir = KFOLD_DIR / f"fold_{fold}"
    ckpt = model_dir / ARCH / "best_model.keras"
    if ckpt.exists():
        print(f"[skip] fold {fold} — checkpoint at {ckpt}")
        continue
    if MAX_FOLDS_PER_RUN is not None and trained_this_run >= MAX_FOLDS_PER_RUN:
        print(f"[stop] MAX_FOLDS_PER_RUN={MAX_FOLDS_PER_RUN} reached — re-run to continue")
        break

    print(f"\n{'=' * 60}\n  training fold {fold}/{K}\n{'=' * 60}")
    cmd = [
        sys.executable, "-m", "scripts.train_single",
        "--arch", ARCH,
        "--epochs", str(EPOCHS),
        "--model-dir", str(model_dir),
        "--log-dir", str(KFOLD_LOG_DIR / f"fold_{fold}"),
        "--nll",
        "--loss-name", "laplace_nll",
        "--nll-beta", str(settings.NLL_BETA),
        "--fold", str(fold),
        "--kfold-k", str(K),
    ]
    subprocess.run(cmd, cwd=project_root, check=True)
    trained_this_run += 1

    hist = json.loads((model_dir / ARCH / "history.json").read_text())
    print(f"\nfold {fold}: best val_loss = {min(hist['val_loss']):.4f}  "
          f"({len(hist['val_loss'])} epochs)")

print(f"\ntrained {trained_this_run} fold(s) this run")

[skip] fold 0 — checkpoint at /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_0/attention_unet_nll/best_model.keras
[skip] fold 1 — checkpoint at /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_1/attention_unet_nll/best_model.keras

  training fold 2/3
k-fold: fold 2/3  train=950  val(held-out)=214

  Architecture: attention_unet_nll (laplace_nll, beta=0.5)


2026-08-27 21:58:15.595191: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-08-27 21:58:15.595207: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-08-27 21:58:15.595212: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-08-27 21:58:15.595223: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-27 21:58:15.595232: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "attention_unet_nll"
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_layer           │ (None, None,      │           0 │ -                  │
│ (InputLayer)          │ None, 3)          │             │                    │
├───────────────────────┼──────��────────────┼─────────────┼────────────────────┤
│ conv2d (Conv2D)       │ (None, None,      │       1,728 │ input_layer[0][0]  │
│                       │ None, 64)         │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ group_normalization   │ (None, None,      │         128 │ conv2d[0][0]       │
│ (GroupNormalization)  │ None, 64)         │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤

/opt/miniconda3/envs/deep-layers/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-08-27 21:58:17.921499: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.2920 - mae: 0.1764 - psnr: 15.4932 - ssim: 0.2954
Epoch 1: val_loss improved from None to -0.39523, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/attention_unet_nll/best_model.keras

Epoch 1: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/attention_unet_nll/best_model.keras
119/119 ━━━━━━━━━━━━━━━━━━━━ 175s 1s/step - loss: -0.2920 - mae: 0.1764 - psnr: 15.4932 - ssim: 0.2954 - val_loss: -0.3952 - val_mae: 0.1346 - val_psnr: 17.1623 - val_ssim: 0.3433 - learning_rate: 1.0000e-04
Epoch 2/100
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: -0.3653 - mae: 0.1350 - psnr: 17.9012 - ssim: 0.4222
Epoch 2: val_loss improved from -0.39523 to -0.40683, saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/kfold/fold_2/attention_unet_nll/best_model.keras

Epoch 2: finished saving model to /Users/simonecaglio/Documents/GitHub/deep_layers/models/k

## 3. Status

In [5]:
print(f"{'fold':<6}{'checkpoint':<12}{'epochs':<9}{'best val_loss':<14}held-out artworks")
print("-" * 90)
for fold in range(K):
    hp = KFOLD_DIR / f"fold_{fold}" / ARCH / "history.json"
    ck = (KFOLD_DIR / f"fold_{fold}" / ARCH / "best_model.keras").exists()
    if hp.exists():
        h = json.loads(hp.read_text())
        ep, bvl = len(h["val_loss"]), min(h["val_loss"])
        print(f"{fold:<6}{'ok' if ck else 'MISSING':<12}{ep:<9}{bvl:<14.4f}{held_out[fold]}")
    else:
        print(f"{fold:<6}{'-':<12}{'-':<9}{'-':<14}{held_out[fold]}")

done = sum((KFOLD_DIR / f'fold_{f}' / ARCH / 'best_model.keras').exists() for f in range(K))
print(f"\n{done}/{K} folds trained." + ("  -> run 061_kfold_evaluation.ipynb" if done >= 2 else "  -> train more before evaluating"))

fold  checkpoint  epochs   best val_loss held-out artworks
------------------------------------------------------------------------------------------
0     ok          22       -0.3844       ['mano', 'mod', 'natmorta2', 'q2', 'q3', 'santo', 'sch02', 'sch03', 'volto']
1     ok          38       -0.4451       ['a1', 'c1', 'cristo', 'natmorta3', 'oblato_tot', 'oblato_volto', 'sch01', 'testa']
2     ok          24       -0.4424       ['a2', 'b1', 'corpo', 'natmorta1', 'orecchio', 'q1', 'torso', 'veste']

3/3 folds trained.  -> run 061_kfold_evaluation.ipynb


## 4. Result (2026-08-27) — all 3 folds trained

Ran across a single session in the end (each fold early-stopped well before the
100-epoch cap). Fold plan is deterministic in `(KFOLD_K=3, KFOLD_SEED=42)`:

| fold | held-out artworks | val / train pairs | epochs | best `val_loss` |
|---|---|---|---|---|
| 0 | `mano` `mod` `natmorta2` `q2` `q3` `santo` `sch02` `sch03` `volto` | 333 / 831 | 22 | −0.3844 |
| 1 | `a1` `c1` `cristo` `natmorta3` `oblato_tot` `oblato_volto` `sch01` `testa` | 221 / 943 | 38 | −0.4451 |
| 2 | `a2` `b1` `corpo` `natmorta1` `orecchio` `q1` `torso` `veste` | 214 / 950 | 24 | −0.4424 |

No NaN, no instability — the three §7 training-bug fixes hold on the k-fold splits
too. Fold 0 holds out the largest / hardest group of artworks and lands ~0.06 higher
in `val_loss` than folds 1–2, which agree tightly.

Checkpoints in `models/kfold/fold_<i>/attention_unet_nll/best_model.keras`
(filesystem only — `models/` is gitignored). Evaluation and the finding-#4 verdict
are in `061_kfold_evaluation.ipynb` (§6): fold-to-fold AUROC std ≈ 0.008, ensemble
`structural z` AUROC 0.72 — the single-split metrics used elsewhere in the project
are confirmed trustworthy.
